# Code mẫu — Optimizer

Optimizer biến gradient thành một bước cập nhật, và nhiều optimizer còn nhớ lịch sử của các bước trước. Trên những hàm loss nhỏ, ta kiểm được công thức, vẽ được đường đi, và soi được trạng thái mà optimizer đang giữ. Hiểu những điều này trước rồi mới chọn cấu hình cho một mạng thật.

## Ví dụ 1: Cùng một gradient, SGD và Adam bước khác nhau

Với $J(w)=(w-3)^2$ và $w=2$, gradient bằng −2. SGD với learning rate 0.1 đưa $w$ tới $2-0.1\cdot(-2)=2.2$. Adam thì chuẩn hóa bước đi, nên bước đầu tiên của nó dài đúng bằng learning rate, đưa $w$ tới 2.1. Ta tính tay bước Adam rồi so với thư viện.

In [ ]:
import math
import torch

torch.set_num_threads(1)
w, gradient, lr = 2., -2., .1
beta1, beta2, eps = .9, .999, 1e-8
m = (1 - beta1) * gradient
v = (1 - beta2) * gradient ** 2
m_hat, v_hat = m / (1 - beta1), v / (1 - beta2)
hand_adam = w - lr * m_hat / (math.sqrt(v_hat) + eps)
print("SGD tính tay:", w - lr * gradient)
print("Adam m, v:", round(m, 6), round(v, 6))
print("Adam m_hat, v_hat:", round(m_hat, 6), round(v_hat, 6))

p = torch.nn.Parameter(torch.tensor(w, dtype=torch.float64))
optimizer = torch.optim.Adam([p], lr=lr, betas=(beta1, beta2), eps=eps)
optimizer.zero_grad(set_to_none=True)
(p - 3).square().backward()
optimizer.step()
print("Adam thư viện / tính tay:", round(p.item(), 6), round(hand_adam, 6))
assert math.isclose(p.item(), hand_adam, rel_tol=1e-12)

SGD tính tay: 2.2
Adam m, v: -0.2 0.004
Adam m_hat, v_hat: -2.0 4.0
Adam thư viện / tính tay: 2.1 2.1


Sau bước đầu, `m` mới là −0.2 và `v` mới là 0.004, nhỏ hơn thực tế nhiều vì cả hai bắt đầu từ 0. Bước hiệu chỉnh đưa chúng về −2 và 4. Bước của Adam là $0.1\cdot(-2)/\sqrt{4}=-0.1$, nên $w$ tăng đúng 0.1. Hai optimizer nhận cùng một gradient. Chúng khác nhau ở quy tắc biến gradient thành bước đi, chứ không phải ở loss.

## Ví dụ 2: Theo dõi từng tọa độ trong một thung lũng hẹp

Loss $J(w_1,w_2)=w_1^2+8w_2^2$ có dạng một thung lũng hẹp: theo trục thứ hai, thành thung lũng dốc gấp 8 lần. Với SGD, mỗi bước nhân $w_2$ với $1-16\eta$. Learning rate 0.05 cho hệ số 0.2, nên $w_2$ co nhanh về 0. Learning rate 0.15 cho hệ số −1.4, nên $w_2$ đổi dấu liên tục và phình ra. Ta so bốn cấu hình, cùng điểm xuất phát, cùng tám bước.

In [ ]:
import torch

torch.set_num_threads(1)
def trajectory(kind, lr):
    p = torch.nn.Parameter(torch.tensor([-2., 1.], dtype=torch.float64))
    if kind == "momentum":
        optimizer = torch.optim.SGD([p], lr=lr, momentum=.8)
    elif kind == "Adam":
        optimizer = torch.optim.Adam([p], lr=lr)
    else:
        optimizer = torch.optim.SGD([p], lr=lr)
    records = []
    for step in range(9):
        loss = p[0].square() + 8 * p[1].square()
        if step in [0, 1, 2, 8]:
            records.append((step, *[round(x, 5) for x in p.detach().tolist()],
                            round(loss.item(), 5)))
        if step < 8:
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
    return records

for kind, lr in [("SGD", .05), ("momentum", .05), ("Adam", .15), ("SGD", .15)]:
    print(kind, "lr", lr, "(step, w1, w2, loss):", trajectory(kind, lr))

SGD lr 0.05 (step, w1, w2, loss): [(0, -2.0, 1.0, 12.0), (1, -1.8, 0.2, 3.56), (2, -1.62, 0.04, 2.6372), (8, -0.86093, 0.0, 0.74121)]
momentum lr 0.05 (step, w1, w2, loss): [(0, -2.0, 1.0, 12.0), (1, -1.8, 0.2, 3.56), (2, -1.46, -0.6, 5.0116), (8, 0.58743, -0.152, 0.52991)]
Adam lr 0.15 (step, w1, w2, loss): [(0, -2.0, 1.0, 12.0), (1, -1.85, 0.85, 9.2025), (2, -1.70042, 0.70112, 6.82401), (8, -0.83365, -0.08203, 0.74881)]
SGD lr 0.15 (step, w1, w2, loss): [(0, -2.0, 1.0, 12.0), (1, -1.4, -1.4, 17.64), (2, -0.98, 1.96, 31.6932), (8, -0.1153, 14.75789, 1742.37596)]


Đọc cột `w2`. SGD với rate 0.05 đưa nó về 0 chỉ sau vài bước, nhưng `w1` thì đi rất chậm, vì theo trục này thung lũng thoải. SGD với rate 0.15 làm `w2` nhảy −1.4, rồi 1.96, rồi tới 14.76 ở bước 8. Loss tăng vọt dù gradient lúc nào cũng tính đúng. Bước quá dài so với độ dốc là đủ để phân kỳ.

Momentum nhớ hướng của các bước trước, nên ở bước 2 `w2` vượt qua đáy sang −0.6. Bù lại, nó kéo `w1` đi xa hơn hẳn, và có loss thấp nhất sau tám bước. Adam bước gần như đều 0.15 trên cả hai trục ở những bước đầu, bất kể trục nào dốc hơn. Đó là tác dụng của việc chia cho căn của `v`. Bốn dòng này chỉ mô tả bốn cấu hình trên một hàm loss. Chúng chưa đủ để xếp hạng optimizer nào tốt hơn cho một bài thật.

## Ví dụ 3: Gradient bằng 0 khác với chưa có gradient

Ta tách riêng phần weight decay của AdamW. Hai tham số cùng bắt đầu ở 2, và optimizer chưa có moment nào từ trước. Một tham số có gradient là tensor 0. Tham số còn lại có `.grad` là `None`, tức chưa từng có gradient.

In [ ]:
import torch

torch.set_num_threads(1)
with_zero = torch.nn.Parameter(torch.tensor(2., dtype=torch.float64))
with_none = torch.nn.Parameter(torch.tensor(2., dtype=torch.float64))
optimizer = torch.optim.AdamW([with_zero, with_none], lr=.1, weight_decay=.2)
with_zero.grad = torch.zeros_like(with_zero)
assert with_none.grad is None
optimizer.step()
print("gradient 0:", round(with_zero.item(), 6))
print("gradient None:", round(with_none.item(), 6))
print("tham số có optimizer state:", len(optimizer.state))
assert abs(with_zero.item() - 1.96) < 1e-12 and with_none.item() == 2.

gradient 0: 1.96
gradient None: 2.0
tham số có optimizer state: 1


Tham số có gradient 0 vẫn bị weight decay kéo nhỏ lại, thành $2(1-0.1\cdot0.2)=1.96$. Tham số có `.grad` là `None` thì bị optimizer bỏ qua hoàn toàn: nó giữ nguyên 2, và không có trạng thái nào được tạo cho nó. Ở đây optimizer còn mới, nên ta thấy riêng tác dụng của decay. Sau nhiều bước, các moment đã lưu vẫn có thể đẩy tham số đi, kể cả khi gradient mới bằng 0.

## Ví dụ 4: Muốn học tiếp, phải lưu cả optimizer lẫn scheduler

Ta chạy ba bước Adam, kèm StepLR giảm learning rate một nửa sau mỗi hai bước, rồi chụp lại toàn bộ trạng thái vào bộ nhớ. Từ cùng trọng số ấy, ta so ba cách đi bước thứ tư. Cách một là chạy tiếp liên tục. Cách hai là khôi phục đầy đủ từ bản chụp. Cách ba là tạo Adam mới tinh, chỉ đặt đúng learning rate hiện tại.

In [ ]:
import copy
import torch

torch.set_num_threads(1)
def make_run(value):
    p = torch.nn.Parameter(torch.tensor(value, dtype=torch.float64))
    opt = torch.optim.Adam([p], lr=.1)
    sched = torch.optim.lr_scheduler.StepLR(opt, step_size=2, gamma=.5)
    return p, opt, sched

def advance(p, opt, sched):
    rate_used = opt.param_groups[0]["lr"]
    opt.zero_grad(set_to_none=True)
    (p - 3).square().backward()
    opt.step()
    sched.step()  # Mỗi lần gọi ứng với một bước cập nhật đã xong.
    return rate_used

p, opt, sched = make_run(2.)
rates = [advance(p, opt, sched) for _ in range(3)]
snapshot = {"weight": p.item(), "optimizer": copy.deepcopy(opt.state_dict()),
            "scheduler": copy.deepcopy(sched.state_dict())}
current_rate = opt.param_groups[0]["lr"]
advance(p, opt, sched)

restored, restored_opt, restored_sched = make_run(snapshot["weight"])
restored_sched.load_state_dict(snapshot["scheduler"])
restored_opt.load_state_dict(snapshot["optimizer"])
advance(restored, restored_opt, restored_sched)

reset = torch.nn.Parameter(torch.tensor(snapshot["weight"], dtype=torch.float64))
reset_opt = torch.optim.Adam([reset], lr=current_rate)
reset_opt.zero_grad(set_to_none=True)
(reset - 3).square().backward()
reset_opt.step()
print("rate ba bước đầu:", rates)
print("liên tục / khôi phục / Adam mới:",
      *[round(v.item(), 8) for v in [p, restored, reset]])
print("rate kế tiếp liên tục / khôi phục:",
      opt.param_groups[0]["lr"], restored_opt.param_groups[0]["lr"])
assert torch.allclose(p, restored, atol=1e-12, rtol=0)
assert not torch.allclose(p, reset, atol=1e-6, rtol=0)

rate ba bước đầu: [0.1, 0.1, 0.05]
liên tục / khôi phục / Adam mới: 2.29806624 2.29806624 2.29900075
rate kế tiếp liên tục / khôi phục: 0.025 0.025


Learning rate là 0.1 ở hai bước đầu, rồi giảm còn 0.05. Bản khôi phục đầy đủ cho trọng số và learning rate kế tiếp trùng khớp với bản chạy liên tục. Bản Adam mới có cùng trọng số và cùng learning rate, nhưng kết quả vẫn lệch. Nó thiếu các moment `m`, `v` và bộ đếm bước, nên hiệu chỉnh của nó tính như đang ở bước đầu tiên.

Ở đây mỗi bước dùng toàn bộ dữ liệu, nên chỉ cần lưu chừng ấy. Khi train bằng minibatch ngẫu nhiên, muốn tiếp tục đúng như đường chạy cũ, ta còn phải lưu cách chia dữ liệu, vị trí đang lấy batch, và trạng thái của bộ sinh số ngẫu nhiên.